# 01 — Exploratory Data Analysis (EDA)
## MURA v1.1: Musculoskeletal Radiograph Abnormality Detection

**Course:** AIE231 — Neural Networks  
**Project:** MURA Abnormality Triage  
**Checkpoint:** 2 — Baselines Ready

---

### Notebook Outline
1. Problem Statement & Metric Justification
2. Dataset Overview
3. Class Distribution Analysis
4. Body-Part Distribution
5. Image Properties (sizes, aspect ratios, pixel intensity)
6. Sample Visualisation
7. Data Quality Checks (duplicates, missing values)
8. Summary of Findings → Modelling Decisions
9. Preprocessing Pipeline Validation (leak-free splits)
10. Baseline Results

In [ ]:
# ── Imports ─────────────────────────────────────────────
import os, sys, pathlib, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

# Add project root to path so we can import src/
PROJECT_ROOT = pathlib.Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import build_dataframe, clean_dataframe, patient_stratified_split, compute_weights

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

## 1. Problem Statement

**Who is the user?** A radiologist triaging musculoskeletal X-rays.  
**What does the model predict?** Whether a radiograph study is *normal* or *abnormal* (binary classification).  
**What does a wrong prediction cost?**
- **False Negative** (missed abnormality): The patient's fracture or pathology is not flagged, delaying treatment. This is the **more dangerous** error in a screening setting.
- **False Positive** (healthy flagged as abnormal): The patient undergoes unnecessary follow-up imaging; costly but not life-threatening.

**Primary metric: Cohen's Kappa (κ)**  
The original MURA paper (Rajpurkar et al., 2017) uses Cohen's Kappa to measure agreement between model and radiologist, which accounts for class imbalance. We additionally report **recall** (sensitivity) because missing an abnormality is the costlier error, and **macro-F1** for balanced per-class performance.

**Input:** 224 × 224 × 3 RGB image (resized from variable original sizes).  
**Output:** Single sigmoid unit → probability of abnormality.  
**Loss:** Binary cross-entropy (with class weights to emphasise abnormal class).

## 2. Dataset Overview

In [ ]:
# ── Configure the dataset path ───────────────────────────
# Change this to wherever you extracted MURA-v1.1
DATA_ROOT = os.path.join(str(PROJECT_ROOT), 'data', 'MURA-v1.1')

# If running locally with a different path, override here:
# DATA_ROOT = r'C:\Users\bdbdr\Downloads\archive\MURA-v1.1'

print(f'Data root: {DATA_ROOT}')
print(f'Exists: {os.path.isdir(DATA_ROOT)}')

In [ ]:
# ── Build image-level dataframe ──────────────────────────
df_train = build_dataframe(DATA_ROOT, 'train_image_paths.csv')
df_valid = build_dataframe(DATA_ROOT, 'valid_image_paths.csv')

print(f'Training images : {len(df_train):,}')
print(f'Validation images: {len(df_valid):,}')
print(f'Total images     : {len(df_train) + len(df_valid):,}')
print()
print(f'Unique patients (train): {df_train["patient_id"].nunique():,}')
print(f'Unique studies  (train): {df_train["study_id"].nunique():,}')
print(f'Unique patients (valid): {df_valid["patient_id"].nunique():,}')
print(f'Unique studies  (valid): {df_valid["study_id"].nunique():,}')

df_train.head()

## 3. Class Distribution

In [ ]:
# ── Image-level class distribution ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, df) in zip(axes, [('Train', df_train), ('Validation', df_valid)]):
    counts = df['label'].value_counts().sort_index()
    bars = ax.bar(['Normal (0)', 'Abnormal (1)'], counts.values,
                  color=['#4C72B0', '#DD8452'])
    ax.set_title(f'{name} — Image-Level Class Distribution')
    ax.set_ylabel('Number of Images')
    for bar, val in zip(bars, counts.values):
        pct = val / counts.sum() * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../experiments/eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTrain label counts:')
print(df_train['label'].value_counts())
print(f'\nImbalance ratio (normal/abnormal): {df_train["label"].value_counts()[0] / df_train["label"].value_counts()[1]:.2f}')

**Finding 3.1:** The training set has approximately 61.5 % normal and 38.5 % abnormal images — a **mild class imbalance** (~1.6:1 ratio). This is not extreme but still requires handling.  

**→ Modelling decision:** Use `class_weight='balanced'` during training to prevent the model from being biased towards the majority class (normal).

## 4. Body-Part Distribution

In [ ]:
# ── Images per body part ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, df) in zip(axes, [('Train', df_train), ('Validation', df_valid)]):
    bp = df.groupby(['body_part', 'label']).size().unstack(fill_value=0)
    bp.columns = ['Normal', 'Abnormal']
    bp = bp.sort_values('Normal', ascending=True)
    bp.plot(kind='barh', stacked=True, ax=ax, color=['#4C72B0', '#DD8452'])
    ax.set_title(f'{name} — Images per Body Part')
    ax.set_xlabel('Number of Images')
    ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('../experiments/eda_body_part_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Per-body-part imbalance
bp_train = df_train.groupby(['body_part', 'label']).size().unstack(fill_value=0)
bp_train.columns = ['Normal', 'Abnormal']
bp_train['Abnormal_pct'] = (bp_train['Abnormal'] / bp_train.sum(axis=1) * 100).round(1)
bp_train['Total'] = bp_train['Normal'] + bp_train['Abnormal']
print(bp_train.sort_values('Total', ascending=False))

**Finding 4.1:** Wrist, shoulder, and hand dominate the dataset; humerus and forearm have far fewer studies.  
**Finding 4.2:** The abnormality rate varies across body parts (e.g., finger studies tend to have a higher abnormality rate than wrist).  

**→ Modelling decision:** Body part is not used as an explicit feature (the model sees only the image), but this variation should be monitored in per-body-part error analysis.

## 5. Image Properties

In [ ]:
# ── Sample image dimensions ──────────────────────────────
sample_paths = df_train['abs_path'].sample(500, random_state=42).tolist()
widths, heights, aspects = [], [], []

for p in sample_paths:
    try:
        with Image.open(p) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            aspects.append(w / h)
    except Exception:
        pass

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(widths, bins=40, color='#4C72B0', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')

axes[1].hist(heights, bins=40, color='#DD8452', edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')

axes[2].hist(aspects, bins=40, color='#55A868', edgecolor='white')
axes[2].set_title('Aspect Ratio Distribution')
axes[2].set_xlabel('Width / Height')

plt.tight_layout()
plt.savefig('../experiments/eda_image_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Width  — min: {min(widths)}, max: {max(widths)}, median: {int(np.median(widths))}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, median: {int(np.median(heights))}')
print(f'Aspect — min: {min(aspects):.2f}, max: {max(aspects):.2f}, median: {np.median(aspects):.2f}')

**Finding 5.1:** Images have **highly variable sizes** — widths range from ~160 px to >4000 px.  
**Finding 5.2:** Aspect ratios are not uniform, so naïve resizing will distort some images.  

**→ Modelling decision:** Resize all images to 224×224 (standard for pretrained CNNs). Accept minor distortion as a trade-off for batch processing. Augmentation (rotation, shifts) will help the model generalise despite this.

In [ ]:
# ── Pixel intensity distribution ─────────────────────────
sample_small = df_train['abs_path'].sample(100, random_state=42).tolist()
means, stds = [], []

for p in sample_small:
    try:
        with Image.open(p) as img:
            arr = np.array(img.convert('L'), dtype=np.float32) / 255.0
            means.append(arr.mean())
            stds.append(arr.std())
    except Exception:
        pass

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(means, bins=30, color='#4C72B0', edgecolor='white')
axes[0].set_title('Mean Pixel Intensity (per image)')
axes[0].set_xlabel('Mean (0–1 scale)')

axes[1].hist(stds, bins=30, color='#DD8452', edgecolor='white')
axes[1].set_title('Std of Pixel Intensity (per image)')
axes[1].set_xlabel('Std')

plt.tight_layout()
plt.savefig('../experiments/eda_pixel_intensity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean intensity — avg: {np.mean(means):.3f}, std: {np.std(means):.3f}')
print(f'Std  intensity — avg: {np.mean(stds):.3f}, std: {np.std(stds):.3f}')

**Finding 5.3:** Pixel intensities vary considerably across images (different exposure levels, contrast).  

**→ Modelling decision:** Rescale pixels to [0, 1] by dividing by 255. For pretrained models, use the standard ImageNet preprocessing.

## 6. Sample Visualisation

In [ ]:
# ── Show sample images per body part and label ───────────
body_parts = sorted(df_train['body_part'].unique())
fig, axes = plt.subplots(len(body_parts), 4, figsize=(14, 3 * len(body_parts)))

for row, bp in enumerate(body_parts):
    subset = df_train[df_train['body_part'] == bp]
    for col, label in enumerate([0, 0, 1, 1]):
        label_subset = subset[subset['label'] == label]
        if len(label_subset) > col // 2:
            img_path = label_subset.iloc[col % 2]['abs_path']
            try:
                img = Image.open(img_path).convert('RGB').resize((224, 224))
                axes[row, col].imshow(img)
            except Exception:
                axes[row, col].text(0.5, 0.5, 'N/A', ha='center')
        axes[row, col].axis('off')
        if row == 0:
            label_name = 'Normal' if label == 0 else 'Abnormal'
            axes[row, col].set_title(f'{label_name} #{col % 2 + 1}')
    axes[row, 0].set_ylabel(bp.replace('XR_', ''), fontsize=11, rotation=0, labelpad=60, va='center')

plt.suptitle('Sample Images — Normal vs Abnormal per Body Part', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../experiments/eda_sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

**Finding 6.1:** Normal and abnormal images can look very similar to the untrained eye, especially for subtle fractures. This confirms the task is non-trivial.  
**Finding 6.2:** Some images have text overlays (R/L markers, hospital labels) which are noise the model must learn to ignore.

## 7. Data Quality Checks

In [ ]:
# ── Duplicates and missing values ────────────────────────
df_clean, report = clean_dataframe(df_train.copy())

print('Cleaning Report:')
print(f"  Duplicate rows removed : {report['duplicates_removed']}")
print(f"  Missing values per col : {report['missing_values']}")
print(f"  Sample files not found : {report['sample_missing_files']} / 500")
print(f"  Clean dataset size     : {len(df_clean):,} images")

**Finding 7.1:** No duplicate image paths detected.  
**Finding 7.2:** No missing values in the metadata columns.  
**Finding 7.3:** All sampled image files exist on disk.  

**→ Modelling decision:** No special cleaning needed beyond standard preprocessing. The dataset is well-curated.

In [ ]:
# ── Images per study distribution ────────────────────────
imgs_per_study = df_train.groupby('study_id').size()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(imgs_per_study, bins=range(1, imgs_per_study.max() + 2), color='#4C72B0', edgecolor='white')
ax.set_title('Number of Images per Study')
ax.set_xlabel('Images in Study')
ax.set_ylabel('Number of Studies')
plt.tight_layout()
plt.savefig('../experiments/eda_images_per_study.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Images per study — min: {imgs_per_study.min()}, max: {imgs_per_study.max()}, '
      f'median: {imgs_per_study.median():.0f}, mean: {imgs_per_study.mean():.1f}')

**Finding 7.4:** Most studies contain 2–3 images (different views of the same body part). Some have up to 10+.  

**→ Modelling decision:** We classify at the **image level** (each image gets its own prediction). Study-level aggregation (e.g., majority vote over views) is a potential improvement for Checkpoint 4.

## 8. Summary of Findings → Modelling Decisions

| # | Finding | Modelling Decision |
|---|---------|--------------------|
| 3.1 | Mild class imbalance (61 % normal, 39 % abnormal) | Use `class_weight='balanced'` in training |
| 4.1 | Uneven body-part distribution | Monitor per-body-part metrics in error analysis |
| 4.2 | Abnormality rate varies by body part | Report body-part breakdown in evaluation |
| 5.1 | Highly variable image sizes | Resize all to 224×224 |
| 5.2 | Non-uniform aspect ratios | Accept minor distortion; augmentation helps |
| 5.3 | Variable pixel intensities | Rescale to [0, 1]; augmentation for robustness |
| 6.1 | Subtle visual differences between classes | Deep CNN needed; transfer learning expected to help |
| 6.2 | Text overlays and markers in some images | Model must learn to ignore; augmentation helps |
| 7.1–3 | No duplicates, no missing values, all files exist | No special cleaning needed |
| 7.4 | Multiple images per study | Classify per-image; study-level aggregation as future work |

## 9. Preprocessing Pipeline Validation

We validate the leak-free splitting: all images from the same patient stay in the same split.

In [ ]:
# ── Patient-level stratified split ───────────────────────
train_split, val_split, test_split = patient_stratified_split(
    df_train.copy(), val_frac=0.15, test_frac=0.15, seed=42
)

print('Split sizes (images):')
print(f'  Train : {len(train_split):,}')
print(f'  Val   : {len(val_split):,}')
print(f'  Test  : {len(test_split):,}')
print()

# Verify no patient leakage
train_patients = set(train_split['patient_id'])
val_patients = set(val_split['patient_id'])
test_patients = set(test_split['patient_id'])

assert train_patients.isdisjoint(val_patients), 'LEAKAGE: train ∩ val'
assert train_patients.isdisjoint(test_patients), 'LEAKAGE: train ∩ test'
assert val_patients.isdisjoint(test_patients), 'LEAKAGE: val ∩ test'
print('✓ No patient leakage between splits.')
print(f'  Train patients: {len(train_patients):,}')
print(f'  Val patients  : {len(val_patients):,}')
print(f'  Test patients : {len(test_patients):,}')

In [ ]:
# ── Label distribution per split ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, split_df) in zip(axes, [('Train', train_split), ('Val', val_split), ('Test', test_split)]):
    counts = split_df['label'].value_counts().sort_index()
    total = counts.sum()
    bars = ax.bar(['Normal', 'Abnormal'], counts.values, color=['#4C72B0', '#DD8452'])
    ax.set_title(f'{name} Split (n={total:,})')
    ax.set_ylabel('Count')
    for bar, val in zip(bars, counts.values):
        pct = val / total * 100
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=9)

plt.suptitle('Label Distribution per Split (Stratified)', fontsize=13)
plt.tight_layout()
plt.savefig('../experiments/eda_split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Class weights ────────────────────────────────────────
cw = compute_weights(train_split)
print(f'Class weights (from training split only):')
print(f'  Normal (0)  : {cw[0]:.4f}')
print(f'  Abnormal (1): {cw[1]:.4f}')
print()
print('These weights are computed from the training split only (no leakage).')
print('They will be passed to model.fit(class_weight=...) to handle imbalance.')

## 10. Baseline Results

We evaluate two baselines required by the specification:
1. **Majority-class baseline** — always predicts the majority class (normal).
2. **Logistic regression baseline** — trained on flattened 64×64 grayscale pixels.

In [ ]:
from src.models import MajorityBaseline, LogisticBaseline, sample_from_generator
from src.data import make_generators
import yaml

# Load config
with open(os.path.join(str(PROJECT_ROOT), 'configs', 'exp_baseline.yaml')) as f:
    cfg = yaml.safe_load(f)

# Override data root
cfg['data']['root_dir'] = DATA_ROOT

# Create generators from our validated splits
train_gen, val_gen, test_gen = make_generators(train_split.copy(), val_split.copy(), test_split.copy(), cfg)

# Sample for sklearn
print('Sampling images for baselines...')
X_train, y_train = sample_from_generator(train_gen, n_samples=3000)
X_val, y_val = sample_from_generator(val_gen, n_samples=1000)

print(f'Sampled — Train: {X_train.shape}, Val: {X_val.shape}')
print(f'Train label dist: {Counter(y_train)}')
print(f'Val   label dist: {Counter(y_val)}')

In [ ]:
# ── Baseline 1: Majority Class ───────────────────────────
majority = MajorityBaseline().fit(X_train, y_train)
majority_metrics = majority.evaluate(X_val, y_val)

print('═══ Majority-Class Baseline ═══')
for k, v in majority_metrics.items():
    if k != 'model':
        print(f'  {k:12s}: {v:.4f}')

In [ ]:
# ── Baseline 2: Logistic Regression ──────────────────────
logistic = LogisticBaseline().fit(X_train, y_train)
logistic_metrics = logistic.evaluate(X_val, y_val)

print('═══ Logistic Regression Baseline ═══')
for k, v in logistic_metrics.items():
    if k != 'model':
        print(f'  {k:12s}: {v:.4f}')

In [ ]:
# ── Comparison Table ─────────────────────────────────────
results_df = pd.DataFrame([majority_metrics, logistic_metrics])
results_df = results_df.set_index('model')
print('\n═══ Baseline Comparison ═══')
print(results_df.round(4).to_string())

# Save
results_df.round(4).to_csv('../experiments/baseline_results.csv')
print('\n✓ Results saved to experiments/baseline_results.csv')

### Baseline Interpretation

- The **majority-class baseline** achieves ~61 % accuracy by always predicting "normal", but has 0 recall for the abnormal class (κ = 0). This is the absolute floor.
- The **logistic regression baseline** provides a simple learned reference point. Any deep network must beat both of these on the validation set to be considered useful.
- Every subsequent model (Checkpoints 3–4) will be reported as an **improvement over these baselines**.

---
*End of EDA — Checkpoint 2 complete.*